# Rangetype Introspection & CQL2-JSON Compound Filter

**Persona: RS Scientist — Remote Sensing Scientist**

## Structure

| Part | Topic |
|------|-------|
| 1 | Band discovery via rangetype (INTROSPECTION capability) |
| 2 | CQL2-JSON compound spatial + attribute filter on coverage features |

## Prerequisites

- A running GeoID / DynaStore instance with:
  - **INTROSPECTION** capability on the target collection's driver (Part 1).
  - **SEARCH** capability for CQL2 filter evaluation (Part 2).
- A collection containing multi-band remote sensing data (e.g. NDVI, rainfall, vegetation health).
- `.env` file with:
  - `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`
- `pip install httpx python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv
import httpx

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN         = os.environ.get("DYNASTORE_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

COV_PREFIX = f"/coverages/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
FEA_PREFIX = f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"

print(f"Catalog    : {CATALOG_ID}")
print(f"Collection : {COLLECTION_ID}")

## Part 1 — Rangetype Band Discovery

The OGC API – Coverages `/coverage/rangetype` endpoint exposes a collection's **spectral/thematic bands** as an array of `field` objects.  
Each field carries its name, unit-of-measure, allowed values, and data type — sufficient for a remote sensing client to build a band-selection UI or automate product interpretation.

This endpoint requires the `INTROSPECTION` capability.  
We first verify the capability at the driver level, then retrieve and parse the rangetype.

In [ ]:
# --- Verify INTROSPECTION capability ---
resp_drv = client.get("/configs/storage/drivers")
assert resp_drv.status_code == 200, (
    f"Driver list failed: {resp_drv.status_code}: {resp_drv.text}"
)

drivers = resp_drv.json()
if isinstance(drivers, dict):
    drivers = list(drivers.values())

print(f"Registered drivers ({len(drivers)}):")
introspection_capable = []
for drv in drivers:
    caps = [c.upper() for c in drv.get("capabilities", [])]
    tags = " ".join(f"[{c}]" for c in caps) or "[no caps]"
    print(f"  {drv.get('driver_id', drv.get('id', '?'))} {tags}")
    if "INTROSPECTION" in caps:
        introspection_capable.append(drv.get("driver_id", drv.get("id", "?")))

if introspection_capable:
    print(f"\nINTROSPECTION-capable drivers: {introspection_capable}")
else:
    print("\nNo driver advertises INTROSPECTION. rangetype will return 501.")

In [ ]:
# --- Retrieve rangetype and parse band names ---
resp_rt = client.get(f"{COV_PREFIX}/coverage/rangetype")
assert resp_rt.status_code in (200, 501), (
    f"rangetype returned unexpected status {resp_rt.status_code}: {resp_rt.text[:300]}"
)

if resp_rt.status_code == 200:
    rt = resp_rt.json()
    fields = rt.get("field", [])
    assert isinstance(fields, list), (
        f"rangeType.field must be a list, got {type(fields).__name__}"
    )
    print(f"Bands ({len(fields)}):")
    band_names = []
    for fld in fields:
        name = fld.get("name", "?")
        uom  = fld.get("uom", {}).get("code", "-")
        dtype = fld.get("dataType", "-")
        defn  = fld.get("definition", "")
        band_names.append(name)
        print(f"  {name:20s}  uom={uom:10s}  dtype={dtype:12s}  {defn}")
    print(f"\nBand name list: {band_names}")
else:
    print("rangetype: HTTP 501 — driver does not implement INTROSPECTION.")

In [ ]:
# --- Behaviour when driver lacks INTROSPECTION ---
# If the driver returned 501 above, document the recommended fallback.
if resp_rt.status_code == 501:
    assert resp_rt.status_code == 501, (
        "Expected 501 for driver without INTROSPECTION"
    )
    print("Expected 501 received — driver correctly advertises lack of INTROSPECTION.")
    print()
    print("Fallback: inspect item properties for band metadata.")

    resp_items = client.get(
        f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
        params={"limit": 1}
    )
    assert resp_items.status_code == 200, (
        f"Fallback item query failed: {resp_items.status_code}: {resp_items.text[:200]}"
    )
    items_data = resp_items.json()
    features = items_data.get("features", [])
    if features:
        props = features[0].get("properties", {})
        eo_bands = props.get("eo:bands", props.get("bands", []))
        print(f"  eo:bands from item properties: {eo_bands}")
    else:
        print("  No items found in collection — cannot demonstrate fallback.")
else:
    print("Driver supports INTROSPECTION — no fallback needed.")

## Part 2 — CQL2-JSON Compound Spatial + Attribute Filter

GeoID's CQL2 pipeline (powered by pygeofilter) supports **compound filters** that combine:

- **Spatial predicates** — e.g. `S_INTERSECTS` against a GeoJSON geometry.
- **Attribute predicates** — e.g. `ndvi_value > 0.4` for vegetation health thresholding.

The filter below targets the **south-western Ethiopia highlands** and retains only items with NDVI above the healthy-vegetation threshold (0.4).

In [ ]:
# --- Build CQL2-JSON compound filter ---
# AOI: south-western Ethiopia highlands
aoi_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [33.0, 3.0],
        [36.0, 3.0],
        [36.0, 6.0],
        [33.0, 6.0],
        [33.0, 3.0]
    ]]
}

filter_body = {
    "op": "and",
    "args": [
        {
            "op": ">",
            "args": [{"property": "ndvi_value"}, 0.4]
        },
        {
            "op": "s_intersects",
            "args": [
                {"property": "geometry"},
                aoi_polygon
            ]
        }
    ]
}

import json
print("CQL2-JSON filter:")
print(json.dumps(filter_body, indent=2))

In [ ]:
# --- Apply filter via features endpoint ---
import urllib.parse

filter_param = json.dumps(filter_body)

resp_feat = client.get(
    f"{FEA_PREFIX}/items",
    params={
        "filter": filter_param,
        "filter-lang": "cql2-json",
        "limit": 20
    }
)
assert resp_feat.status_code == 200, (
    f"CQL2-JSON filter on features failed: {resp_feat.status_code}: {resp_feat.text[:400]}"
)

feat_data = resp_feat.json()
assert "features" in feat_data, (
    f"Features response must contain 'features' key. Keys: {list(feat_data.keys())}"
)

count = feat_data.get("context", {}).get("returned", len(feat_data["features"]))
print(f"Items matching NDVI > 0.4 ∩ S_INTERSECTS(AOI): {count}")
for feat in feat_data["features"][:5]:
    props = feat.get("properties", {})
    ndvi  = props.get("ndvi_value", "?")
    dt    = props.get("datetime", "?")
    print(f"  id={feat['id']}  ndvi={ndvi}  datetime={dt}")

In [ ]:
# --- Same CQL2-JSON filter via coverages endpoint ---
# Not all drivers implement filtered coverage retrieval; 501 is an acceptable fallback.
resp_cov_flt = client.get(
    f"{COV_PREFIX}/coverage",
    params={
        "filter": filter_param,
        "filter-lang": "cql2-json"
    }
)
assert resp_cov_flt.status_code in (200, 501), (
    f"Filtered coverage returned unexpected status {resp_cov_flt.status_code}: "
    f"{resp_cov_flt.text[:300]}"
)

if resp_cov_flt.status_code == 200:
    ct = resp_cov_flt.headers.get("content-type", "")
    print(f"Filtered coverage: HTTP 200  Content-Type={ct}  size={len(resp_cov_flt.content)} bytes")
else:
    print("Filtered coverage: HTTP 501 — driver does not support CQL2 filtering on coverage endpoint.")
    print("Use the features endpoint (cell above) to apply CQL2 filters and retrieve item metadata instead.")

## Edge Cases

The CQL2 parser must reject:

1. An **invalid GeoJSON polygon** (unclosed ring or wrong coordinate count) — expect **HTTP 400**.
2. A **mixed filter language** mismatch (body says `cql2-json` but `filter-lang` says `cql2-text`) — expect **HTTP 400**.

In [ ]:
# --- Invalid GeoJSON polygon: ring not closed ---
bad_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [33.0, 3.0],
        [36.0, 3.0],
        [36.0, 6.0]
        # Missing [33.0, 6.0] and closing [33.0, 3.0] — ring is not closed
    ]]
}

bad_filter = {
    "op": "s_intersects",
    "args": [
        {"property": "geometry"},
        bad_polygon
    ]
}

resp_bad_geom = client.get(
    f"{FEA_PREFIX}/items",
    params={
        "filter": json.dumps(bad_filter),
        "filter-lang": "cql2-json"
    }
)
assert resp_bad_geom.status_code == 400, (
    f"Invalid polygon should return 400, got {resp_bad_geom.status_code}: {resp_bad_geom.text[:300]}"
)
print(f"Invalid GeoJSON polygon → HTTP {resp_bad_geom.status_code} (expected 400)")

In [ ]:
# --- filter-lang mismatch: JSON body but text declared ---
# The filter value is a CQL2-JSON object, but filter-lang claims cql2-text
mismatch_filter = json.dumps({
    "op": ">",
    "args": [{"property": "ndvi_value"}, 0.4]
})

resp_mismatch = client.get(
    f"{FEA_PREFIX}/items",
    params={
        "filter": mismatch_filter,
        "filter-lang": "cql2-text"   # wrong lang for a JSON object payload
    }
)
assert resp_mismatch.status_code == 400, (
    f"filter-lang mismatch should return 400, got {resp_mismatch.status_code}: {resp_mismatch.text[:300]}"
)
print(f"filter-lang mismatch → HTTP {resp_mismatch.status_code} (expected 400)")